# Detect Sources

This guide shows how to detect sources in an observation using wavelet techniques. The same approach can also be used to initialize sources for scene modeling. For this demonstration, we'll closely follow the [Real-World Example](../0-example).

In [ ]:
# Import Packages and setup
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scarlet2 as sc2

# use a suitable colormap and don't interpolate the pixels
import matplotlib
matplotlib.rc('image', interpolation='none', origin="lower", cmap="inferno")

## Data Loading

We load the example data set (a HSC 5-band image). It contains a detection catalog, but we will initially pretend it does not.

In [ ]:
from huggingface_hub import hf_hub_download

filename = hf_hub_download(
    repo_id="astro-data-lab/scarlet-test-data", filename="hsc_cosmos_35.npz", repo_type="dataset"
)
file = jnp.load(filename)
data = file["images"]
channels = [str(f) for f in file["filters"]]
catalog = jnp.array([(src["y"], src["x"]) for src in file["catalog"]])  # Note: y/x convention!
weights = 1 / file["variance"]
psf = file["psfs"]

While some of the detection methods below work just fine on regular image arrays, working with {py:class}`~scarlet2.Observation` make many operations more convenient.

In [ ]:
obs = sc2.Observation(data, weights=weights, psf=psf, channels=channels)

# display the observation
norm = sc2.plot.AsinhAutomaticNorm(obs)
sc2.plot.observation(obs, norm=norm);

## Starlets

We employ the starlet transformation, popularized by [Starck, Murtagh & Bertero (2015)](https://doi.org/10.1007/978-0-387-92920-0_34), implemented in {py:func}`scarlet2.wavelets.starlet_transform`, to decompose the observation into different scales.
 In detail, the method {py:func}`scarlet2.detect.get_detect_wavelets` combines all bands of the HSC image cube with inverse-variance weights into a detection image, which then gets transformed.
 At every scale, the resulting starlet coefficients form an image of the same shape as the original, but with increasingly larger spatial scales.
 The last scale contains everything that is not contained in smaller scales, which means that the transformation is exact (in fact, it's overcomplete because there are multiple ways to represent the same input image).
 The method returns the coefficients with significant support across multiple scales.
 In this filtered image, called `detect` below, areas above some threshold should correspond to the sources we seek to find. We then run a flood-filling algorithm to find connected regions of a certain size: source footprints.

In [ ]:
scales = [0,1,2,3]
# for strict scale separation: run detection only on filtered scales
# add another scale for the remainder
strict = True
max_scale = max(scales) + strict

# multi-scale support threshold K=2 means s 2-sigma fluctuation across bands and scales
K=2
detect, sigma = sc2.detect.get_detect_wavelets(obs.data, variance=1/obs.weights, max_scale=max_scale, K=K)

# find compact regions of a minimum area above 0 in detection image:
# footprints of detected source, shown as semi-transparent contours
min_area=9
im_shape = detect[0].shape
def add_footprints(ax, _detect):
    footprint_map = jnp.zeros(im_shape)
    fps = sc2.detect.footprints(_detect, min_area=min_area)
    for fp in fps:
        # footprints have binary array `footprint`
        # and `bounds` to describe the position of `footprint` in `detect`
        # so we can insert the footprint array in the larger image
        footprint_map += sc2.bbox.insert_into(
            jnp.zeros(im_shape),
            fp.footprint,
            sc2.Box.from_bounds(*fp.bounds)
        )
        # footprints also contain local peaks
        for peak in fp.peaks:
            ax.scatter(peak.x, peak.y, marker='x', color="w")
    ax.imshow(footprint_map, cmap="grey", alpha=0.3)

# show each detection scale, compact footprints and peaks
w,h = 3,4
fig, axes = plt.subplots(1, max_scale+1, figsize = ((max_scale+1)*w,h))
for scale, ax in enumerate(axes):
    ax.imshow(detect[scale])
    if scale < max_scale:
        ax.set_title(f"scale:{scale}")
        add_footprints(ax, detect[scale])
    else:
        ax.set_title(f"scale:{scale}+")
    ax.axis('off')
fig.tight_layout()
plt.show()

We can see that smaller scales allow smaller sources to be found, even in the presence of neighbors, whereas larger scales capture extended source emission.

```{tip}

Reducing `K` lets the algorithm find fainter sources, but makes it more vulnerable to noise fluctuations. Increasing `min_area` removes small fluctuations, but can reject small legitimate sources, too.
When in doubt, run the code above on your data and tweak `scales`, `K`, and `min_area` until you are satisfied.
```

It's clear that one can run detection at any of these scales (in fact, we've just done so): find connected regions, aka "footprints", above some threshold, then find peaks within footprints. But a good detection method should be able to combine information _across scales_ to deal with compact and extended sources.

## Hierarchical Detection

For that, we have created a hierarchical detection method, {py:func}`scarlet2.detect.hierarchical_footprints`, which starts with the footprints found on the largest scale and adds those of all smaller scales. If a footprint on a smaller scale overlaps with a footprint on a larger scale but is offset from the larger-scale peak, it is added as child of the larger footprint. Footprints within footprints... Which makes a lot of sense for blended source.

In [ ]:
footprints = sc2.detect.hierarchical_footprints(
    detect,
    scales=scales,
    sigma_scales=sigma,
    K=K,
    min_area=min_area,
)
sc2.plot.observation(
    obs,
    norm=norm,
    add_footprints=footprints, # show footprints
    add_peaks=footprints,      # show footprint peaks
    label_kwargs={"color": "red"}
);

That looks very reasonable. Let's have a closer look at one of the footprints:

In [ ]:
footprints[0]

Each footprint contains the binary array `footprint` that indicates the pixels above the detection threshold, a bounding box `bbox` to place that array in the original image, the primary `peak`, the `scale` it was detected on, and, possibly, `children`.

But there are no children here, even though it looks like sources 0 and 1 should be in the same footprint on scale 3, so where's the hierarchy? By default, the hierarchy is flattened and peaks in the same footprints are split into separate sources by a watershed algorithm, which means that we get footprints like this:

In [ ]:
footprint_map = jnp.zeros(im_shape)
for i,fp in enumerate(footprints):
    footprint_map += sc2.bbox.insert_into(
        jnp.zeros(im_shape),
        (i+1)*fp.footprint,
        fp.bbox
    )
plt.imshow(footprint_map)
plt.show()

Having non-overlapping footprints is reasonable for un- or weakly resolved sources like most of those in this observation. If we want to fully acknowledge the possible overlap between sources, we can prevent the flattening and peak splitting:

In [ ]:
footprints = sc2.detect.hierarchical_footprints(
    detect,
    scales=scales,
    sigma_scales=sigma,
    K=K,
    min_area=min_area,
    split_peaks=False,
    flatten=False,
)
print(footprints[0])

footprint_map = jnp.zeros(im_shape)
for i,fp in enumerate(footprints):
    footprint_map += sc2.bbox.insert_into(
        jnp.zeros(im_shape),
        (i+1)*fp.footprint,
        fp.bbox
    )
plt.imshow(footprint_map)
plt.show()

Source 0 now contains what we used to call source 1 as its child, and its footprint covers both. The same happens with the smaller sources 2 and 4.

So, we now have a list (either hierarchical or flattened) of detected sources with their peaks and outlines from a multi-scale filtering method.

## Hierarchical Initialization

Finding good guesses for sources in crowded or blended scenes is hard. But the hierarchical footprints allow us to go back and extract sources from the observation. The method {py:func}`scarlet.init.hierarchical_sources` performs the same starlet filtering and hierarchical detection as shown above, and then goes through all scales (from the largest to the smallest) and removes the contributions of each source within its footprint.

In [ ]:
model_frame = sc2.Frame.from_observations(obs)

with sc2.Scene(model_frame) as scene:
    infos = sc2.init.hierarchical_sources(obs, K=K, min_area=min_area)
    for peak, center, spectrum, morph in infos:
        sc2.Source(center, spectrum, morph)

Let's check what we have in `scene`:

In [ ]:
sc2.plot.scene(
    scene,
    obs,
    norm=norm,
    show_model=True,
    show_rendered=True,
    show_observed=True,
    add_boxes=True,
);

Not too bad...

### Initialization with Given Catalog

Often we already have a detection catalog from some previous processing stage (in fact, we have one for this HSC example). So, {py:func}`scarlet2.detect.hierarchical_footprints` and {py:func}`scarlet.init.hierarchical_sources` accept an argument `catalog` with RA/DEC coordinates. When provided, the methods will go through the catalog list and report only footprints/sources at those locations, or `None`. Let's demonstrate:

In [ ]:
with sc2.Scene(model_frame) as scene:
    infos = sc2.init.hierarchical_sources(
        obs,
        K=K,
        min_area=min_area,
        catalog=catalog
    )
    for peak, center, spectrum, morph in infos:
        sc2.Source(center, spectrum, morph)

sc2.plot.scene(
    scene,
    obs,
    norm=norm,
    show_model=True,
    show_rendered=True,
    show_observed=True,
    add_boxes=True,
);

You can see that the order of some sources has changed, and that what used to be called "source 5" on the right edge of the image is not initialized because it is not in the provided detection catalog.
But we can now proceed with [fitting the scene](../0-example).